In [37]:
import pandas as pd


In [38]:
data = pd.read_csv(
    "/Users/rahulkulkarni/Desktop/personal project/DATA SCIENTIST/online_retail_II.csv",
    encoding="ISO-8859-1",
)
print(data.shape)
print(data.dtypes)
print(data.head())

(1067371, 8)
Invoice         object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
Price          float64
Customer ID    float64
Country         object
dtype: object
  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

           InvoiceDate  Price  Customer ID         Country  
0  2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
3  2009-12-01 07:45:00   2.10      13085.0  United Kingdom  
4  2009-12-01 07:45:00   1.25      13085.0  United Kingdom  


In [39]:
print(data["Description"].str.contains("Ã|Â|�", na=False, regex=True).sum())


79


# missing values

In [40]:
print(data.isnull().sum())
print(data.isnull().mean() * 100)

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64
Invoice         0.000000
StockCode       0.000000
Description     0.410541
Quantity        0.000000
InvoiceDate     0.000000
Price           0.000000
Customer ID    22.766873
Country         0.000000
dtype: float64


## Date Range Check

In [41]:
data["InvoiceDate"] = pd.to_datetime(data["InvoiceDate"])
print(data["InvoiceDate"].min(), data["InvoiceDate"].max())

2009-12-01 07:45:00 2011-12-09 12:50:00


Q2: Why does the date range matter specifically for calculating "Recency" in RFM — what reference point do you need, and why can't you just use today's real-world date?

Here's the reasoning: Recency is "how many days since this customer's last purchase." That's a relative measurement — it needs a fixed point to count backward from. The natural choice is the max date in the dataset (2011-12-09), or one day after it, used as a stand-in for "today" within the context of this dataset.

If you instead used today's real-world date (2026), every single customer's Recency would be enormous — the "most recent" customer, who bought something on 2011-12-09, would show a Recency of roughly 5,300+ days. That number would be technically correct but completely useless for segmentation, because every customer would look equally "long gone," and you'd lose all ability to distinguish an actively engaged customer from a truly inactive one — the entire point of Recency as a signal would collapse.

So the rule is: Recency's reference point should be relative to the dataset's own timeframe, not the real-world calendar, unless you're specifically building something that updates live against real time (which this static, historical dataset isn't).

## Uniqueness Counts

In [42]:
print("unique customers:", data["Customer ID"].nunique())
print("unique invoices:", data["Invoice"].nunique())
print("unique products (stockcode):", data["StockCode"].nunique())
print("unique countries:", data["Country"].nunique())
print(data["Country"].value_counts().head(10))

unique customers: 5942
unique invoices: 53628
unique products (stockcode): 5305
unique countries: 43
Country
United Kingdom    981330
EIRE               17866
Germany            17624
France             14330
Netherlands         5140
Spain               3811
Switzerland         3189
Belgium             3123
Portugal            2620
Australia           1913
Name: count, dtype: int64


Given that 92% of the data is UK-only and the remaining 8% is spread thin across 42 other countries, would you (a) drop non-UK rows entirely, (b) keep everything and segment globally anyway, or (c) something else?

A3: I'd keep all rows and all real country labels (not collapse non-UK countries into a generic "Other" bucket), and run a single RFM + clustering model across the full customer base — since Country isn't actually used in the RFM calculation itself (Recency/Frequency/Monetary don't depend on it), it doesn't need to affect how the model is built. I would not build separate models per country, since most non-UK countries have too few customers (some in the single digits) to cluster meaningfully on their own. After clustering, I'd report Country as a secondary breakdown — e.g., what % of each segment is UK vs. other countries — and note in the write-up that ~92% of the data is UK-based, so results should be understood as primarily reflecting UK customer behavior.

## Quantity Distribution & Anomalies

In [43]:
print(data["Quantity"].describe())
print("Negative quantity rows:", (data["Quantity"] < 0).sum())
print("Zero quantity rows:", (data["Quantity"] == 0).sum())


count    1.067371e+06
mean     9.938898e+00
std      1.727058e+02
min     -8.099500e+04
25%      1.000000e+00
50%      3.000000e+00
75%      1.000000e+01
max      8.099500e+04
Name: Quantity, dtype: float64
Negative quantity rows: 22950
Zero quantity rows: 0


Q4: The mean (9.94) and median (3.0) are quite different from each other. What does that difference tell you about the shape of this distribution, and why would you generally prefer median over mean as the "typical" value when a dataset looks like this?

A4: The gap between mean and median signals a right-skewed distribution — a small number of extreme outliers (like the 80,995-unit bulk order we found) are pulling the mean upward, even though most transactions are actually small (the median of 3 reflects what a typical order really looks like). This happens because mean is calculated by summing every value and dividing by count, so every value — including extreme outliers — pulls on it equally. Median only depends on position (the middle value when sorted), not magnitude, so it's resistant to outliers. In this dataset, using mean Quantity to describe a "typical customer" would be misleading, since it's distorted by a handful of wholesale-scale orders that don't represent most customers' actual behavior.

## Price Distribution & Anomalies

In [44]:
print(data["Price"].describe())
print("Negative price rows:", (data["Price"] < 0).sum())
print("Zero price rows:", (data["Price"] == 0).sum())


count    1.067371e+06
mean     4.649388e+00
std      1.235531e+02
min     -5.359436e+04
25%      1.250000e+00
50%      2.100000e+00
75%      4.150000e+00
max      3.897000e+04
Name: Price, dtype: float64
Negative price rows: 5
Zero price rows: 6202


Q5: If most of the zero-price rows have a legitimate-looking product description (not "damaged," "test," "lost," etc.) — does that change your decision about whether to drop them? What would you want to check before deciding?

A5: Yes — you can't apply one blanket rule to all zero-price rows, because they're not a single category. Looking at a sample, most zero-price rows fall into two groups: internal warehouse/inventory noise (descriptions like "short," "lost," "damages," "mixed," or unclear codes), and legitimate charge types like DOTCOM POSTAGE priced at $0 in a specific row. Almost all of these also have a missing Customer ID, meaning they'd already be removed by the first cleaning step regardless. The exception is rows with a real product description AND a real Customer ID (e.g., "6 RIBBONS EMPIRE" with Customer ID 16126.0) — these survive the missing-ID filter and need their own explicit decision. Before dropping any zero-price rows, I'd filter specifically to Price == 0 AND Customer ID not null, to see how many rows actually remain after the first cleaning step, rather than assuming the full 6,202 count still applies. Since Monetary is based on real revenue, a $0 transaction contributes nothing to it either way, so dropping the remaining ones is still reasonable — but the count needs to be verified, not assumed.

## cancelled invoices

In [45]:
cancelled = data["Invoice"].astype(str).str.startswith("C")
print("Cancelled invoice rows:", cancelled.sum())
print(cancelled.mean() * 100, "% of rows")

Cancelled invoice rows: 19494
1.8263565339511754 % of rows


You now have three different "negative quantity"/anomaly patterns in this dataset: (1) cancelled invoices — Invoice starts with "C", (2) internal warehouse adjustments — negative quantity + $0 price + no Customer ID, and (3) "Adjust bad debt" rows — StockCode "B", negative price, no Customer ID. If you only used the "C-prefix" rule to filter out negative quantities, what would you miss, and why does that matter for your final cleaned dataset?

A6: Filtering only on the "C" prefix would correctly catch the 19,494 customer-initiated cancellations, but it would miss the ~3,457 internal warehouse-adjustment rows and the 5 "Adjust bad debt" rows, since neither of those patterns is reflected in the invoice number — they'd slide straight through into the "cleaned" dataset undetected. In this specific dataset, it turns out not to matter much in practice, because nearly all of those rows also have a missing Customer ID, so they get removed anyway by the very first cleaning step (dropping missing Customer ID) before the cancellation filter even runs. But that's a coincidence of this particular dataset, not something to rely on generally — if even one adjustment row had a real Customer ID attached, a C-prefix-only filter would silently let bad data corrupt that customer's RFM numbers with no warning. The lesson is not to assume one filter catches everything just because it handles the obvious case — each anomaly pattern needs to be checked independently and verified against what's already being removed, rather than assumed to be covered.

## duplicates

In [46]:
dupes = data[data.duplicated(keep=False)]
print(dupes.sort_values("Invoice").head(20))


    Invoice StockCode                        Description  Quantity  \
362  489517     21913     VINTAGE SEASIDE JIGSAW PUZZLES         1   
394  489517     21912           VINTAGE SNAKES & LADDERS         1   
391  489517     21491    SET OF THREE VINTAGE GIFT WRAPS         1   
390  489517    84951A    S/4 PISTACHIO LOVEBIRD COASTERS         1   
388  489517    84951A    S/4 PISTACHIO LOVEBIRD COASTERS         1   
386  489517     21821   GLITTER STAR GARLAND WITH BELLS          1   
385  489517     21913     VINTAGE SEASIDE JIGSAW PUZZLES         1   
384  489517     22319  HAIRCLIPS FORTIES FABRIC ASSORTED        12   
379  489517     21491    SET OF THREE VINTAGE GIFT WRAPS         1   
371  489517     21912           VINTAGE SNAKES & LADDERS         1   
368  489517     22130   PARTY CONE CHRISTMAS DECORATION          6   
367  489517     22319  HAIRCLIPS FORTIES FABRIC ASSORTED        12   
365  489517     21821   GLITTER STAR GARLAND WITH BELLS          1   
363  489517     2191

You found that "duplicate" rows share the same Invoice number and appear as scattered individual line items rather than the entire invoice duplicated as a block. Does it matter whether you drop these duplicates, and why?

A7: Yes, it matters a lot — and the initial assumption that "exact duplicates should be dropped" turned out to be wrong for this dataset. If duplicates were a true data export glitch, you'd expect the entire invoice to be logged twice, in the same relative order. Instead, individual line items within the same invoice (same Invoice number, same customer, same timestamp) repeat inconsistently — some products appear once, others two or three times. This pattern is more consistent with a customer genuinely adding the same product to their cart multiple times as separate entries, or the system splitting a quantity into multiple rows, rather than an export error. Since these rows represent real purchased units and real revenue, running .drop_duplicates() would silently delete legitimate quantity and undercount that customer's true Frequency and Monetary values. Decision: keep these rows and do not run .drop_duplicates() on the dataset, documenting the investigation and reasoning rather than applying a blanket rule based on an untested assumption.

## Non-Product Stock Codes

In [47]:
non_numeric = data[~data["StockCode"].astype(str).str.match(r"^\d")]
print(non_numeric["StockCode"].value_counts().head(20))


StockCode
POST            2122
DOT             1446
M               1421
C2               282
D                177
S                104
BANK CHARGES     102
ADJUST            67
AMAZONFEE         43
DCGS0058          31
gift_0001_20      29
gift_0001_30      29
DCGSSGIRL         25
DCGSSBOY          23
PADS              19
gift_0001_10      16
CRUK              16
TEST001           15
DCGS0076          15
DCGS0003          14
Name: count, dtype: int64


Given the list of non-product stock codes (POST, DOT, M, C2, D, S, BANK CHARGES, ADJUST, AMAZONFEE, DCGS codes, gift cards, CRUK, TEST001), would you write one blanket rule ("drop everything that's not purely numeric") or would you need multiple, different rules for different code types? What's the risk of using just one simple rule here?

A8: One blanket rule would be a mistake here — the non-numeric codes aren't a single category, they represent at least three different things: legitimate customer-paid revenue (POST, DOT, C2 for postage/carriage; gift_0001_xx for gift cards), pure accounting/administrative noise that should be dropped (M, D, BANK CHARGES, ADJUST, AMAZONFEE, TEST001, CRUK), and ambiguous cases that need direct investigation before deciding (S for samples — likely $0 price and not a real paid transaction; the DCGS-prefixed codes, which don't match the standard numeric pattern but may still be real products with non-standard SKUs). If I dropped everything non-numeric with one rule, I'd incorrectly remove real revenue like postage charges, which would undercount customers' true Monetary values. The risk of a single blanket rule is treating "doesn't match my regex" as equivalent to "not a real transaction," when in reality the pattern only tells you a code is unusual, not why it's unusual — each type needs to be checked individually against its actual meaning before deciding whether it belongs in the cleaned dataset.

## Cleaning

In [48]:
before = len(data)
data = data[data["Customer ID"].notnull()]
print(f"Dropped {before - len(data)} rows -> {len(data)} remaining")


Dropped 243007 rows -> 824364 remaining


In [49]:
before = len(data)
data = data[~data["Invoice"].astype(str).str.startswith("C")]
print(f"Dropped {before - len(data)} rows -> {len(data)} remaining")


Dropped 18744 rows -> 805620 remaining


In [50]:
before = len(data)
admin_codes = ["M", "D", "BANK CHARGES", "ADJUST", "AMAZONFEE", "TEST001", "CRUK"]
data = data[~data["StockCode"].isin(admin_codes)]
print(f"Dropped {before - len(data)} rows -> {len(data)} remaining")

Dropped 796 rows -> 804824 remaining


In [51]:
print(
    data[data["StockCode"] == "S"][
        ["StockCode", "Description", "Quantity", "Price"]
    ].head(10)
)


Empty DataFrame
Columns: [StockCode, Description, Quantity, Price]
Index: []


In [52]:
print(
    data[data["StockCode"].astype(str).str.startswith("DCGS")][
        ["StockCode", "Description"]
    ]
    .drop_duplicates()
    .head(10)
)


Empty DataFrame
Columns: [StockCode, Description]
Index: []


In [53]:
print(
    f"FINAL: {len(data)} rows | {data['Customer ID'].nunique()} customers | {data['Invoice'].nunique()} invoices"
)
data.to_csv("cleaned_retail.csv", index=False)


FINAL: 804824 rows | 5855 customers | 36718 invoices


## RFM Feature Engineering

In [54]:
import duckdb

In [55]:
data = pd.read_csv("cleaned_retail.csv")
data["InvoiceDate"] = pd.to_datetime(data["InvoiceDate"])

print(data.shape)


(804824, 8)


In [56]:
con = duckdb.connect()
con.register("retail", data)

con.sql("select * from retail limit 5").show()

┌─────────┬───────────┬─────────────────────────────────────┬──────────┬─────────────────────┬────────┬─────────────┬────────────────┐
│ Invoice │ StockCode │             Description             │ Quantity │     InvoiceDate     │ Price  │ Customer ID │    Country     │
│  int64  │  varchar  │               varchar               │  int64   │    timestamp_ns     │ double │   double    │    varchar     │
├─────────┼───────────┼─────────────────────────────────────┼──────────┼─────────────────────┼────────┼─────────────┼────────────────┤
│  489434 │ 85048     │ 15CM CHRISTMAS GLASS BALL 20 LIGHTS │       12 │ 2009-12-01 07:45:00 │   6.95 │     13085.0 │ United Kingdom │
│  489434 │ 79323P    │ PINK CHERRY LIGHTS                  │       12 │ 2009-12-01 07:45:00 │   6.75 │     13085.0 │ United Kingdom │
│  489434 │ 79323W    │  WHITE CHERRY LIGHTS                │       12 │ 2009-12-01 07:45:00 │   6.75 │     13085.0 │ United Kingdom │
│  489434 │ 22041     │ RECORD FRAME 7" SINGLE SIZE    

In [ ]:
con.sql("""
select "Customer ID",sum(Quantity * Price) as Monetary from retail
group by "Customer ID"
order by Monetary Desc
limit 10""").show()

┌─────────────┬────────────────────┐
│ Customer ID │      Monetary      │
│   double    │       double       │
├─────────────┼────────────────────┤
│     18102.0 │  608821.6499999999 │
│     14646.0 │  528602.5199999996 │
│     14156.0 │ 305228.62999999995 │
│     14911.0 │ 284029.61000000074 │
│     17450.0 │ 246973.08999999994 │
│     13694.0 │ 196482.81000000003 │
│     17511.0 │ 175603.55000000002 │
│     16446.0 │           168472.5 │
│     16684.0 │ 147142.77000000002 │
│     12415.0 │          144383.37 │
└─────────────┴────────────────────┘
  10 rows                2 columns



Customer 18102 has a Monetary value of $608,821.65 — dramatically higher than most customers (compare to the ones at the bottom of your earlier full 5,855-row output, many in the hundreds of dollars). Is a number this extreme likely to be a data error, a legitimate wholesale/business buyer, or something else? What would you check before deciding whether to treat this row specially in your clustering step later?

In [58]:
con.sql("""
SELECT
    Invoice,
    StockCode,
    Description,
    Quantity,
    Price,
    InvoiceDate
FROM retail
WHERE "Customer ID" = 18102
ORDER BY InvoiceDate
""").show()


┌─────────┬───────────┬─────────────────────────────────────┬──────────┬────────┬─────────────────────┐
│ Invoice │ StockCode │             Description             │ Quantity │ Price  │     InvoiceDate     │
│  int64  │  varchar  │               varchar               │  int64   │ double │    timestamp_ns     │
├─────────┼───────────┼─────────────────────────────────────┼──────────┼────────┼─────────────────────┤
│  489438 │ 21329     │ DINOSAURS  WRITING SET              │       28 │   0.98 │ 2009-12-01 09:24:00 │
│  489438 │ 21252     │ SET OF MEADOW  FLOWER STICKERS      │       30 │   1.69 │ 2009-12-01 09:24:00 │
│  489438 │ 21100     │ CHARLIE AND LOLA CHARLOTTE BAG      │       30 │   1.15 │ 2009-12-01 09:24:00 │
│  489438 │ 21033     │ JUMBO BAG CHARLIE AND LOLA TOYS     │       30 │    2.0 │ 2009-12-01 09:24:00 │
│  489438 │ 20711     │ JUMBO BAG TOYS                      │       60 │    1.3 │ 2009-12-01 09:24:00 │
│  489438 │ 21410     │ COUNTRY COTTAGE  DOORSTOP GREEN     │   

How many separate invoices, and over what time span?

In [59]:
con.sql("""
select
count(Distinct Invoice) as num_invoices,
MIN(InvoiceDate) as first_purchase,
MAX(InvoiceDate) as last_purchase,
COUNT(*) AS num_line_items
from retail
where "Customer Id" = 18102
""").show()

┌──────────────┬─────────────────────┬─────────────────────┬────────────────┐
│ num_invoices │   first_purchase    │    last_purchase    │ num_line_items │
│    int64     │    timestamp_ns     │    timestamp_ns     │     int64      │
├──────────────┼─────────────────────┼─────────────────────┼────────────────┤
│          145 │ 2009-12-01 09:24:00 │ 2011-12-09 11:50:00 │           1058 │
└──────────────┴─────────────────────┴─────────────────────┴────────────────┘



What country are they in, and does the product mix look consistent (like buying the same few items repeatedly, which is common for a business reselling stock) or scattered/random?

In [60]:
con.sql("""
select
Country,StockCode,Description,Sum(Quantity) as total_qty
from retail
where "Customer ID" = 18102
group by Country,StockCode,Description
order by total_qty Desc
Limit 10
""").show()

┌────────────────┬───────────┬───────────────────────────────────┬───────────┐
│    Country     │ StockCode │            Description            │ total_qty │
│    varchar     │  varchar  │              varchar              │  int128   │
├────────────────┼───────────┼───────────────────────────────────┼───────────┤
│ United Kingdom │ 22189     │ CREAM HEART CARD HOLDER           │     11324 │
│ United Kingdom │ 22188     │ BLACK HEART CARD HOLDER           │      9938 │
│ United Kingdom │ 82484     │ WOOD BLACK BOARD ANT WHITE FINISH │      6001 │
│ United Kingdom │ 21623     │ VINTAGE UNION JACK MEMOBOARD      │      5168 │
│ United Kingdom │ 21877     │ HOME SWEET HOME MUG               │      3712 │
│ United Kingdom │ 21872     │ GLAMOROUS  MUG                    │      2902 │
│ United Kingdom │ 48194     │ DOORMAT HEARTS                    │      2460 │
│ United Kingdom │ 22507     │ MEMO BOARD RETROSPOT  DESIGN      │      2360 │
│ United Kingdom │ 21871     │ SAVE THE PLANET MUG  

So to answer the question - I'd check the number of distinct invoices, the time span of their activity, and the pattern of products purchased. For Customer 18102, I found 145 distinct invoices spanning almost the full 2-year dataset, with consistent repeated bulk purchases of home-décor items (thousands of units of the same products) — this rules out a one-time data error and strongly suggests a wholesale/reseller account rather than an individual consumer. This matters for clustering because the project's core question is about individual customer loyalty and retention, and wholesale purchasing patterns reflect business restocking cycles, not consumer engagement — mixing them into the same K-means model would either isolate wholesale accounts into a meaningless single-customer cluster or distort distance calculations for everyone else, since K-means is sensitive to scale. Decision: identify wholesale-scale accounts using a defensible threshold (e.g., unusually high total Monetary and/or average order size) and treat them as a separate category rather than clustering them together with individual retail customers.

In [61]:
con.sql("""
SELECT
    "Customer ID",
    SUM(Quantity * Price) AS Monetary
FROM retail
GROUP BY "Customer ID"
""").df()["Monetary"].describe()


count      5855.000000
mean       3003.665619
std       14670.476488
min           0.000000
25%         349.025000
50%         898.960000
75%        2303.130000
max      608821.650000
Name: Monetary, dtype: float64

(to revisit and apply on Day 3): Monetary is heavily right-skewed (mean $3,003 vs. median $899, max $608,821). Should raw Monetary values be used directly in K-means, or transformed first (e.g., log transform)? Why?

A10: Transform — because K-means uses distance-based math, and an unscaled feature with this much range (e.g., $0 to $608,821) would dominate the clustering, making Recency and Frequency almost irrelevant by comparison. A log transform (e.g., log(Monetary + 1)) compresses the extreme range while preserving meaningful differences among typical customers, preventing wholesale-scale outliers from distorting the whole model.

In [62]:
con.sql("""
select
"Customer ID",
count(distinct Invoice) as Frequency
from retail
group by "Customer ID"
ORDER BY FREQUENCY DESC
LIMIT 10
""").show()

┌─────────────┬───────────┐
│ Customer ID │ Frequency │
│   double    │   int64   │
├─────────────┼───────────┤
│     14911.0 │       375 │
│     12748.0 │       324 │
│     17841.0 │       211 │
│     15311.0 │       207 │
│     13089.0 │       203 │
│     14606.0 │       185 │
│     17850.0 │       155 │
│     14646.0 │       152 │
│     14156.0 │       145 │
│     18102.0 │       145 │
└─────────────┴───────────┘
  10 rows       2 columns



In [63]:
con.sql("""
SELECT
    "Customer ID",
    COUNT(DISTINCT Invoice) AS Frequency
FROM retail
GROUP BY "Customer ID"
""").df()["Frequency"].describe()


count    5855.000000
mean        6.271221
std        12.799145
min         1.000000
25%         1.000000
50%         3.000000
75%         7.000000
max       375.000000
Name: Frequency, dtype: float64

In [64]:
con.sql("SELECT MAX(InvoiceDate) AS max_date FROM retail").show()


┌─────────────────────┐
│      max_date       │
│    timestamp_ns     │
├─────────────────────┤
│ 2011-12-09 12:50:00 │
└─────────────────────┘



In [65]:
con.sql("""
select
"Customer ID",
date_diff('day',max(InvoiceDate),(select max(InvoiceDate) from retail)) as Recency
from retail
group by "Customer ID"
order by Recency desc
limit 10
""").show()

┌─────────────┬─────────┐
│ Customer ID │ Recency │
│   double    │  int64  │
├─────────────┼─────────┤
│     14654.0 │     738 │
│     17056.0 │     738 │
│     13526.0 │     738 │
│     12636.0 │     738 │
│     17592.0 │     738 │
│     14106.0 │     737 │
│     17606.0 │     737 │
│     17087.0 │     737 │
│     17909.0 │     737 │
│     14980.0 │     737 │
└─────────────┴─────────┘
  10 rows     2 columns



In [66]:
con.sql("""
SELECT
    "Customer ID",
    DATE_DIFF('day', MAX(InvoiceDate), (SELECT MAX(InvoiceDate) FROM retail)) AS Recency
FROM retail
GROUP BY "Customer ID"
""").df()["Recency"].describe()


count    5855.000000
mean      199.794876
std       208.610894
min         0.000000
25%        25.000000
50%        95.000000
75%       379.000000
max       738.000000
Name: Recency, dtype: float64

In [67]:
rfm = con.sql("""
select
"Customer ID",
DATE_DIFF('day', MAX(InvoiceDate), (SELECT MAX(InvoiceDate) FROM retail)) AS Recency,
    COUNT(DISTINCT Invoice) AS Frequency,
    SUM(Quantity * Price) AS Monetary
FROM retail
GROUP BY "Customer ID"
""").df()

print(rfm.shape)
print(rfm.head())

(5855, 4)
   Customer ID  Recency  Frequency   Monetary
0      16329.0      427          4    1703.07
1      14156.0        9        145  305228.63
2      17063.0       21         13    3638.53
3      15793.0       14          8    2399.57
4      14061.0      507          3     917.43


In [68]:
print(rfm.isnull().sum())
print(rfm.describe())


Customer ID    0
Recency        0
Frequency      0
Monetary       0
dtype: int64
        Customer ID      Recency    Frequency       Monetary
count   5855.000000  5855.000000  5855.000000    5855.000000
mean   15318.455167   199.794876     6.271221    3003.665619
std     1715.393034   208.610894    12.799145   14670.476488
min    12346.000000     0.000000     1.000000       0.000000
25%    13835.500000    25.000000     1.000000     349.025000
50%    15319.000000    95.000000     3.000000     898.960000
75%    16801.500000   379.000000     7.000000    2303.130000
max    18287.000000   738.000000   375.000000  608821.650000


In [69]:
# Monetary
data["Revenue"] = data["Quantity"] * data["Price"]
monetary_pd = data.groupby("Customer ID")["Revenue"].sum()

# Frequency
frequency_pd = data.groupby("Customer ID")["Invoice"].nunique()

# Recency
max_date = data["InvoiceDate"].max()
last_purchase = data.groupby("Customer ID")["InvoiceDate"].max()
recency_pd = (max_date - last_purchase).dt.days

# Combine into one DataFrame
rfm_pandas = pd.DataFrame(
    {"Recency": recency_pd, "Frequency": frequency_pd, "Monetary": monetary_pd}
).reset_index()

print(rfm_pandas.shape)
print(rfm_pandas.head())


(5855, 4)
   Customer ID  Recency  Frequency  Monetary
0      12346.0      325          4  77353.96
1      12347.0        1          8   5633.32
2      12348.0       74          5   2019.40
3      12349.0       18          4   4428.69
4      12350.0      309          1    334.40


In [70]:
comparison = rfm.merge(rfm_pandas, on="Customer ID", suffixes=("_sql", "_pandas"))

comparison["Recency_diff"] = comparison["Recency_sql"] - comparison["Recency_pandas"]
comparison["Frequency_diff"] = (
    comparison["Frequency_sql"] - comparison["Frequency_pandas"]
)
comparison["Monetary_diff"] = comparison["Monetary_sql"] - comparison["Monetary_pandas"]

print(comparison[["Recency_diff", "Frequency_diff", "Monetary_diff"]].describe())


       Recency_diff  Frequency_diff  Monetary_diff
count   5855.000000          5855.0   5.855000e+03
mean       0.533903             0.0  -4.727536e-14
std        0.498892             0.0   1.258719e-11
min        0.000000             0.0  -4.656613e-10
25%        0.000000             0.0  -2.842171e-14
50%        1.000000             0.0   0.000000e+00
75%        1.000000             0.0   1.421085e-14
max        1.000000             0.0   7.566996e-10


## Day 2: RFM Feature Engineering — Summary

### What was built
Computed Recency, Frequency, and Monetary per customer, two independent ways:
1. SQL (via duckdb, run directly against the cleaned dataset)
2. Pandas (groupby-based), as a cross-check

### Final RFM table
- 5,855 rows (one per customer), 4 columns: Customer ID, Recency, Frequency, Monetary
- Zero missing values across all columns

### Key distributions (all 5,855 customers)
| Metric | Mean | Median | Max | Notes |
|---|---|---|---|---|
| Recency (days) | 199.8 | 95 | 738 | Less skewed than F/M; bounded by dataset's 2-year span |
| Frequency (# invoices) | 6.27 | 3 | 375 | Heavily right-skewed; 25% of customers ordered only once |
| Monetary ($) | 3,003.67 | 898.96 | 608,821.65 | Heavily right-skewed; driven by a small number of wholesale-scale accounts |

### Key finding: wholesale/B2B accounts exist in the data
Investigated the top Monetary/Frequency outliers (Customer 18102 and 14156) directly:
- 145+ distinct invoices each, spanning nearly the full 2-year dataset
- Consistent, repeated bulk purchases of the same product categories (thousands of units)
- Conclusion: these are very likely wholesale/reseller accounts, not individual consumers
- Decision (to apply on Day 3): since the project's core question is about individual customer retention, wholesale-scale accounts should be identified via a defensible threshold and treated as a separate category rather than clustered together with retail customers

### Validation: SQL vs. pandas cross-check
- Frequency: exact match across all 5,855 customers (0 difference)
- Monetary: matched to within floating-point rounding error (~1e-10, not a real discrepancy)
- Recency: systematic 0-or-1-day difference across roughly half of customers — under investigation (see Q11)

### Parked decisions for Day 3 (clustering)
- Log-transform Recency, Frequency, and Monetary before clustering, since all are right-skewed and K-means is distance-based (an untransformed Monetary of $608K would dominate the model)
- Decide on a threshold to separate wholesale-scale accounts from individual retail customers before/during clustering

In [71]:
# Pick one customer where SQL and pandas disagreed, and look at the actual timestamps
sample_customer = comparison[comparison["Recency_diff"] == 1]["Customer ID"].iloc[0]
print("Customer ID:", sample_customer)
print(
    "Their last purchase timestamp:",
    data[data["Customer ID"] == sample_customer]["InvoiceDate"].max(),
)
print("Dataset max timestamp:", data["InvoiceDate"].max())


Customer ID: 17063.0
Their last purchase timestamp: 2011-11-18 16:19:00
Dataset max timestamp: 2011-12-09 12:50:00


Q11: The Recency values from SQL and pandas differed by exactly 0 or 1 day for roughly half of customers — never a fraction, never anything else. What explains this systematic gap, and which version should be used as final?

A11: SQL's DATE_DIFF('day', ...) compares only the calendar-date portion of two timestamps, ignoring time-of-day entirely — so Nov 18 to Dec 9 is calculated as a flat 21 days regardless of the specific times involved. Pandas' .dt.days (from subtracting two full timestamps) calculates the exact elapsed time in hours, then floors to whole days — so the same Nov 18 4:19pm to Dec 9 12:50pm gap comes out as 20 days, since the partial final day hadn't fully elapsed yet by clock time. Neither approach is incorrect, but they represent two different definitions of "days since last purchase." Decision: use SQL's calendar-date version as final, since RFM segmentation operates at day-level business granularity, not hour-level precision — two customers who both purchased "on November 18th" shouldn't get different Recency values just because one checked out in the morning and the other at night. This also matches the common convention used in most RFM implementations.

In [72]:
rfm.to_csv("rfm_table.csv", index=False)
print("Saved:", rfm.shape)


Saved: (5855, 4)
